# Solution of  Linear Systems of Equations

The HHL algorithm (based on surnames Harrow-Hassidim-Lloyd) - quantum algorithm to solve systems of linear equations 
$$A \vec{x} = \vec{b}$$.

To execute this algorithm, we require 4 main steps with three qubit registers:
<br><b>Step 1 :</b> Express the vector $\vec{b}$ as a quantum state $|b\rangle$ on a quantum register
<br><b>Step 2 :</b> Decompose $\vec{b}$ into a superposition of eigenvectors of $A$ remembering on the linear combination of the vector $\vec{b}$. We achieve this using the Quantum Phase Estimation algorithm (Quantum Phase Estimation (QPE)). Since the matrix is hereby diagonalized wherefore $A$ is easily invertible.
<br><b>Step 3 :</b> The inversion of the eigenvector base of $A$ is achieved by rotating an ancillary qubit by an angle $\arcsin \left( \frac{C}{\lambda _{\text{i}}} \right)$ around the y-axis where $\lambda_{\text{i}}$ are the eigenvalues of $A$. Now, we obtain the state $A^{-1}|b\rangle = |x \rangle$.
<br><b>Step 4 :</b> We need to uncompute the register storing the eigenvalues using the inverse QPE. We measure the ancillary qubit whereby the measurement of 1 indicates that the matrix inversion was successful. The inverse QPE leaves the system in a state proportional to the solution vector $|x\rangle$. In many cases one is not interested in the single vector elements of $|x\rangle$ but only on certain properties. These are accessible by applying a problem-specific operator $M$ to the state $|x\rangle$. 

Another use-case of the HHL algorithm is the implementation in a larger quantum program. 
</ol> Currently only hermitian matrices with a dimension of $2^n$ are supported.

Take into account that in the general case, the entries of $\vec{x}$ can not be efficiently read out because we would need to know all coefficients describing the quantum state. In the following examples, we ignore this constraint and show for our small linear system as a proof of principle that $\vec{x}$ is calculated correctly.

In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
import numpy as np
from scipy.sparse import diags
from qiskit_backend_config import BasicAer, execute
# qiskit.algorithms.linear_solvers (HHL, NumPyLinearSolver, TridiagonalToeplitz) was removed from
# Qiskit and is not part of qiskit-algorithms. HHL is built here from basic circuit parts and wrapped
# in the same small interface the cells below use.
import numpy as np
from scipy.linalg import expm
from qiskit import QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit.library import QFTGate, RYGate, StatePreparation, UnitaryGate
from qiskit.quantum_info import Statevector, state_fidelity


def matrix_resize(matrix, vector):
    """Pad to a power-of-two size and make the system Hermitian (as Aqua's HHL.matrix_resize did)."""
    matrix, vector = np.asarray(matrix, dtype=complex), np.asarray(vector, dtype=complex)
    truncate_hermitian = not np.allclose(matrix, matrix.conj().T)
    if truncate_hermitian:   # [[0, A], [A^dagger, 0]] (x', x) = (b, 0)
        n = len(matrix)
        matrix = np.block([[np.zeros((n, n)), matrix], [matrix.conj().T, np.zeros((n, n))]])
        vector = np.concatenate([vector, np.zeros(n)])
    size = 1 << int(np.ceil(np.log2(len(matrix))))
    truncate_powerdim = size != len(matrix)
    if truncate_powerdim:
        pad = size - len(matrix)
        matrix = np.block([[matrix, np.zeros((len(matrix), pad))], [np.zeros((pad, len(matrix))), np.eye(pad)]])
        vector = np.concatenate([vector, np.zeros(pad)])
    return matrix, vector, truncate_powerdim, truncate_hermitian


class HHL:
    """Harrow-Hassidim-Lloyd linear solver built from basic circuit parts.

    Registers: |b> (log2 N qubits), clock (num_ancillae [+1 sign bit]) and one rotation ancilla.
    1. prepare |b>   2. QPE of exp(iAt)   3. controlled rotations RY(2 arcsin(C/lambda))
    4. uncompute the QPE; post-select ancilla = 1, clock = 0 -> |x> ~ A^-1 |b>
    """

    def __init__(self, matrix, vector, num_ancillae=3, negative_evals=False, scale=1.0, orig_size=None):
        self._matrix = np.asarray(matrix, dtype=complex)
        self._vector = np.asarray(vector, dtype=complex)
        self._orig_size = orig_size or len(vector)
        self._negative_evals = negative_evals
        self._num_clock = num_ancillae + (1 if negative_evals else 0)
        # evolution time: put the smallest |eigenvalue| exactly on the QPE grid if the largest
        # one still fits in the clock register, otherwise map the largest one to the top value
        evals = np.abs(np.linalg.eigvalsh(self._matrix))
        lambda_min, lambda_max = evals.min(), evals.max()
        top = 2 ** (self._num_clock - (1 if negative_evals else 0)) - 1   # largest magnitude index
        j_min = int(lambda_min * top / lambda_max + 1e-7)
        if j_min >= 1:
            self._evo_time = 2 * np.pi * j_min / (2 ** self._num_clock * lambda_min)
        else:
            self._evo_time = 2 * np.pi * top / (2 ** self._num_clock * lambda_max)
        self._lambda_res = 2 * np.pi / (self._evo_time * 2 ** self._num_clock)
        # constant C in C / lambda: the smallest eigenvalue the clock register can represent
        self._c = scale * self._lambda_res * max(1, round(lambda_min / self._lambda_res))
        self.circuit = self._construct_circuit()

    def _construct_circuit(self):
        nb = int(np.log2(len(self._matrix)))
        b = QuantumRegister(nb, 'b')
        clock = QuantumRegister(self._num_clock, 'clock')
        anc = QuantumRegister(1, 'anc')
        qc = QuantumCircuit(b, clock, anc, name='HHL')
        qc.append(StatePreparation(self._vector / np.linalg.norm(self._vector)), b)

        qpe = QuantumCircuit(b, clock, name='QPE')
        qpe.h(clock)
        for k in range(self._num_clock):
            u_k = UnitaryGate(expm(1j * self._matrix * self._evo_time * 2 ** k), label=f'U^{2 ** k}')
            qpe.append(u_k.control(1), [clock[k]] + list(b))
        qpe.append(QFTGate(self._num_clock).inverse(), clock)
        qc.compose(qpe, qubits=list(b) + list(clock), inplace=True)

        for j in range(1, 2 ** self._num_clock):    # eigenvalue inversion
            signed = j - 2 ** self._num_clock if self._negative_evals and j >= 2 ** (self._num_clock - 1) else j
            angle = 2 * np.arcsin(np.clip(self._c / (signed * self._lambda_res), -1, 1))
            qc.append(RYGate(angle).control(self._num_clock, ctrl_state=j), list(clock) + [anc[0]])

        qc.compose(qpe.inverse(), qubits=list(b) + list(clock), inplace=True)
        return qc

    def postselected_vector(self, backend=None):
        """Amplitudes of |b> when the ancilla is 1 and the clock register is back to 0."""
        nb = self.circuit.qregs[0].size
        if backend is None:
            sv = Statevector(self.circuit).data
        else:   # a statevector simulator backend
            sv = np.asarray(execute(self.circuit, backend).result().get_statevector())
        offset = 1 << (nb + self._num_clock)
        return sv[offset: offset + 2 ** nb]

    def run(self, backend=None):
        vec = self.postselected_vector(backend)
        probability = float(np.real(np.vdot(vec, vec)))
        vec = vec / np.linalg.norm(vec)
        # rescale like Aqua: match |A x| to |b| and fix the global phase
        tmp = self._matrix @ vec
        f1 = np.linalg.norm(self._vector) / np.linalg.norm(tmp)
        nonzero = np.abs(self._vector) > 1e-12
        f2 = np.mean(np.angle(self._vector[nonzero] * tmp[nonzero].conj()))
        solution = (f1 * vec * np.exp(-1j * f2))[: self._orig_size]
        basis = transpile(self.circuit, basis_gates=['u', 'cx'], optimization_level=0)
        return {'solution': solution, 'probability_result': probability,
                'circuit_info': {'width': basis.num_qubits, 'depth': basis.depth()}}


def fidelity(hhl, ref):
    solution_hhl_normed = hhl / np.linalg.norm(hhl)
    solution_ref_normed = ref / np.linalg.norm(ref)
    fidelity = state_fidelity(solution_hhl_normed, solution_ref_normed)
    print("fidelity %f" % fidelity)


_HHLCircuit = HHL


class LinearSolverResult:
    def __init__(self, state, euclidean_norm):
        self.state = state                    # circuit (HHL) or solution vector (NumPy)
        self.euclidean_norm = euclidean_norm  # |x| for A x = b / |b|


class NumPyLinearSolver:
    def solve(self, matrix, vector):
        x = np.linalg.solve(matrix, vector)
        return LinearSolverResult(state=x, euclidean_norm=np.linalg.norm(x))


class TridiagonalToeplitz:
    """Symmetric tridiagonal Toeplitz matrix of size 2^num_state_qubits.

    The removed Qiskit class also supplied a gate-efficient circuit for exp(iAt); that construction
    is not reproduced here, so HHL builds the same circuit as for the dense matrix.
    """
    def __init__(self, num_state_qubits, main_diag, off_diag):
        size = 2 ** num_state_qubits
        self.matrix = diags([off_diag, main_diag, off_diag], [-1, 0, 1], shape=(size, size)).toarray()


class HHL:
    """Same interface as the removed qiskit.algorithms.linear_solvers.HHL: HHL().solve(matrix, vector)."""
    def solve(self, matrix, vector):
        A = matrix.matrix if isinstance(matrix, TridiagonalToeplitz) else np.asarray(matrix)
        vector = np.asarray(vector)
        nb = int(np.log2(len(A)))
        evals = np.linalg.eigvalsh(A)
        kappa = np.max(np.abs(evals)) / np.min(np.abs(evals))
        nl = max(nb + 1, int(np.ceil(np.log2(kappa + 1))))     # clock qubits, as in the old HHL
        hhl = _HHLCircuit(A, vector, num_ancillae=nl, negative_evals=bool(np.any(evals < 0)))
        x = hhl.postselected_vector()                          # = C * A^-1 |b>
        return LinearSolverResult(state=hhl.circuit, euclidean_norm=np.linalg.norm(x) / hhl._c)


In [ ]:
import numpy as np
matrix = np.array([[1, -1/3], [-1/3, 1]])
vector = np.array([1, 0])
naive_hhl_solution = HHL().solve(matrix, vector)

In [ ]:
classical_solution = NumPyLinearSolver().solve(matrix, vector / np.linalg.norm(vector))

In [ ]:
tridi_matrix = TridiagonalToeplitz(1, 1, -1 / 3)

tridi_solution = HHL().solve(tridi_matrix, vector)

In [ ]:
print('classical state:', classical_solution.state)

In [ ]:
print('naive state:')
print(naive_hhl_solution.state)
print('tridiagonal state:')
print(tridi_solution.state)

In [ ]:
print('classical Euclidean norm:', classical_solution.euclidean_norm)
print('naive Euclidean norm:', naive_hhl_solution.euclidean_norm)
print('tridiagonal Euclidean norm:', tridi_solution.euclidean_norm)

In [ ]:
from qiskit.quantum_info import Statevector

naive_sv = Statevector(naive_hhl_solution.state).data
tridi_sv = Statevector(tridi_solution.state).data

# Extract the right vector components. 1000 corresponds to the index 8 and 1001 corresponds to the index 9
naive_full_vector = np.array([naive_sv[8], naive_sv[9]])
tridi_full_vector = np.array([tridi_sv[8], tridi_sv[9]])

print('naive raw solution vector:', naive_full_vector)
print('tridi raw solution vector:', tridi_full_vector)

In [ ]:
naive_full_vector = np.real(naive_full_vector)
tridi_full_vector = np.real(tridi_full_vector)

In [ ]:
print('full naive solution vector:', naive_hhl_solution.euclidean_norm*naive_full_vector/np.linalg.norm(naive_full_vector))
print('full tridi solution vector:', tridi_solution.euclidean_norm*tridi_full_vector/np.linalg.norm(tridi_full_vector))
print('classical state:', classical_solution.state)

In [ ]:
num_qubits = 2
matrix_size = 2 ** num_qubits
# entries of the tridiagonal Toeplitz symmetric matrix
a = 1
b = -1/3

matrix = diags([b, a, b], [-1, 0, 1], shape=(matrix_size, matrix_size)).toarray()
vector = np.array([1] + [0]*(matrix_size - 1))

# run the algorithms
classical_solution = NumPyLinearSolver().solve(matrix, vector / np.linalg.norm(vector))
naive_hhl_solution = HHL().solve(matrix, vector)
tridi_matrix = TridiagonalToeplitz(num_qubits, a, b)
tridi_solution = HHL().solve(tridi_matrix, vector)

print('classical euclidean norm:', classical_solution.euclidean_norm)
print('naive euclidean norm:', naive_hhl_solution.euclidean_norm)
print('tridiagonal euclidean norm:', tridi_solution.euclidean_norm)

*Note:* the efficient `TridiagonalToeplitz` circuit construction of the removed Qiskit class is not reproduced in this notebook, so the naive and the tridiagonal HHL circuits below are identical and their depths match (the original tutorial showed the tridiagonal circuits getting much shallower).

In [ ]:
from qiskit import transpile

num_qubits = list(range(1,5))
a = 1
b = -1/3

i=1
# calculate the circuit depths for different number of qubits to compare the use of resources
naive_depths = []
tridi_depths = []
for nb in num_qubits:
    matrix = diags([b, a, b], [-1, 0, 1], shape=(2**nb, 2**nb)).toarray()
    vector = np.array([1] + [0]*(2**nb -1))
    
    naive_hhl_solution = HHL().solve(matrix, vector)
    tridi_matrix = TridiagonalToeplitz(nb, a, b)
    tridi_solution = HHL().solve(tridi_matrix, vector)

    naive_qc = transpile(naive_hhl_solution.state,basis_gates=['id', 'rz', 'sx', 'x', 'cx'])
    tridi_qc = transpile(tridi_solution.state,basis_gates=['id', 'rz', 'sx', 'x', 'cx'])
    
    naive_depths.append(naive_qc.depth())
    tridi_depths.append(tridi_qc.depth())
    i +=1

In [ ]:
sizes = [str(2**nb)+"x"+str(2**nb) for nb in num_qubits]
columns = ['size of the system', 'quantum_solution depth', 'tridi_solution depth']
data = np.array([sizes, naive_depths, tridi_depths])
row_format ="{:>23}" * (len(columns) + 2)
for team, row in zip(columns, data):
    print(row_format.format(team, *row))

In [ ]:
print('excess:', [naive_depths[i] - tridi_depths[i] for i in range(0, len(naive_depths))])

# References
1. A. W. Harrow, A. Hassidim, and S. Lloyd, Phys. Rev. Lett. 103, 150502 (2009), e-print arXiv 0811.3171<br>
2. S. Barz, I. Kassal, M. Ringbauer, Y. Ole Lipp, B. Dakić, A. Aspuru-Guzik, and P. Walther, Sci Rep. 4: 6115 (2014), e-print arXiv 1302.1210<br>